In [ ]:
import pandas as pd

from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

from catboost import CatBoostClassifier

In [ ]:
data = pd.read_csv('data/train.csv')
data.head()

In [ ]:
class DataCleaner:
    def __init__(self):
        self.columns_to_drop = ['Ticket', 'Cabin', 'Name', 'PassengerId']
    
    def transform(self, data):
        data = data.drop(self.columns_to_drop, axis=1, errors='ignore')
        data = data.dropna()
        return data
    
    def fit(self):
        return self

In [ ]:
cleaner = DataCleaner()
data = cleaner.transform(data)
data

In [ ]:
train_data = data.drop('Survived', axis=1)
train_labels = data[['Survived']]
train_data.head()

In [ ]:
class SimplePreprocesser:
    def __init__(self):
        self.column_transformer = ColumnTransformer(
            [
                ('ohe_encoder', OneHotEncoder(drop='first'), ['Pclass', 'Sex', 'Embarked'])
            ],
            sparse_threshold=0,
            remainder='passthrough'
        )
        self.columns_to_drop = ['Ticket', 'Cabin', 'Name']
    
    def fit(self, data):
        data = self._process_siblings(data)
        self.column_transformer.fit(data)
        return self
    
    def transform(self, data):
        data = self._process_siblings(data)
        return self.column_transformer.transform(data)
    
    @staticmethod
    def _process_siblings(data):
        data['Siblings'] = data['SibSp'] + data['Parch']
        return data.drop(['SibSp', 'Parch'], axis=1)

In [ ]:
processor = SimplePreprocesser()
processor.fit(train_data)
train_data = processor.transform(train_data)
train_data

In [ ]:
model = CatBoostClassifier()
model.fit(train_data, train_labels)

In [ ]:
data['Fare'].hist()

In [ ]:
data_entry = pd.DataFrame({
    'Pclass': [1],
    'Sex': 'male',
    'Age': 23,
    'SibSp': 0,
    'Parch': 0,
    'Fare': 300,
    'Embarked': 'S'
})
data_entry

In [ ]:
processed_string = processor.transform(cleaner.transform(data_entry))

In [ ]:
prediction = model.predict_proba(processed_string)[:, 0] * 100
print(f'Dead with probability of {prediction[0]:.2f}%.')

## Sample to test API

In [ ]:
import requests

url = 'http://127.0.0.1:8000/predict'

data = {
    "PassengerId": 0,
    "Pclass": 3,
    "Name": 'Alex Morgan',
    "Sex": "male",
    "Age": 22,
    "SibSp": 1,
    "Parch": 0,
    'Ticket': 'BX322',
    "Fare": 7.25,
    'Cabin': '32S',
    "Embarked": "S"
}

response = requests.post(url, json=data)

print(response.text)